# Phase 5 — GNN-Based Network Intrusion Detection
## UNSW-NB15 | Edge Classification via Graph Neural Networks

**Task framing:** Edge classification — every network flow (edge `srcip → dstip`) gets a binary label (Normal/Attack) or one of 10 multiclass labels.

| Variant | Conv Layers | Edge Features in MP | Notes |
|---|---|---|---|
| `NIDS-GCN` | GCNConv × 2 | No | Simplest GNN baseline |
| `NIDS-GAT` | GATv2Conv × 2 (4→1 heads) | Yes (attention `edge_dim=64`) | Attention-based |
| `NIDS-SAGE` | SAGEConv × 2 | No | Inductive, scalable |
| `NIDS-FULL` | GATv2Conv(4h) + SAGEConv | Yes | Full proposed model |

All variants share an **edge classifier MLP** tail: `cat[h_src ‖ edge_feat ‖ h_dst] → 256 → 128 → num_classes`

**Evaluation suite:** Standard metrics + calibration + MC dropout uncertainty + node/edge embedding t-SNE + GAT attention analysis + ablations + robustness + gradient attribution.

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Platform selector + package installation
# Set PLATFORM to match your environment before running.
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, os
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

PLATFORM = 'kaggle'   # 'local' | 'kaggle' | 'colab'

# ── OOM safety flag ─────────────────────────────────────────────────────────
# If the FULL model OOMs on your GPU, set REDUCE_HEADS=True (halves attention
# head count and hidden_dim → ~4× less memory at slight accuracy cost).
REDUCE_HEADS = False

# ── Eval suite flag ──────────────────────────────────────────────────────────
# Set to False to skip Cells 26–33 (MC-Dropout, t-SNE, ablations, etc.)
# and jump straight to saving gnn_results.json.  Core training/eval is unaffected.
RUN_EVAL_SUITE = False

if PLATFORM in ('kaggle', 'colab'):
    pkgs = ['torch-geometric', 'captum', 'umap-learn']
    for pkg in pkgs:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', pkg],
            capture_output=True, text=True
        )
        status = '✓' if result.returncode == 0 else '✗'
        print(f'{status} {pkg}')
else:
    print('Local environment — skipping pip installs.')

print(f'Platform: {PLATFORM} | Reduce heads: {REDUCE_HEADS} | Eval suite: {RUN_EVAL_SUITE}')


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 — Path configuration
# ─────────────────────────────────────────────────────────────────────────────
if PLATFORM == 'kaggle':
    BASE    = '/kaggle/input/datasets/dhruvbhadhotiya/unsw-nb15/outputs'
    OUT_DIR = '/kaggle/working/outputs/models'
    OUT_GRF = f'{BASE}/graph'
    OUT_PRE = f'{BASE}/preprocessed'
    OUT_ENC = f'{BASE}/encoders'
    OUT_MDL_IN = f'{BASE}/models'   # read-only: baseline_results.json lives here
    OUT_MDS = '/kaggle/working'   # for outputs.md (writeable)

elif PLATFORM == 'colab':
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print('WARNING: google.colab not available — running outside Colab?')
    BASE    = '/content/drive/MyDrive/UNSW-NB15/outputs'
    OUT_DIR = f'{BASE}/models'
    OUT_GRF = f'{BASE}/graph'
    OUT_PRE = f'{BASE}/preprocessed'
    OUT_ENC = f'{BASE}/encoders'
    OUT_MDL_IN = OUT_DIR   # same dir for colab (read-write Drive)
    OUT_MDS = '/content/drive/MyDrive/UNSW-NB15'

else:  # local
    BASE    = r'c:\Users\Asus\OneDrive\Desktop\GNN\UNSW-NB15'
    OUT_DIR = os.path.join(BASE, 'outputs', 'models')
    OUT_GRF = os.path.join(BASE, 'outputs', 'graph')
    OUT_PRE = os.path.join(BASE, 'outputs', 'preprocessed')
    OUT_ENC = os.path.join(BASE, 'outputs', 'encoders')
    OUT_MDL_IN = OUT_DIR   # same dir locally
    OUT_MDS = BASE

os.makedirs(OUT_DIR, exist_ok=True)
print(f'OUT_DIR    : {OUT_DIR}')
print(f'OUT_MDL_IN : {OUT_MDL_IN}')
print(f'OUT_GRF    : {OUT_GRF}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import gc, json, warnings, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.checkpoint import checkpoint as grad_checkpoint

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATv2Conv, SAGEConv

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss, matthews_corrcoef
)
from sklearn.calibration import calibration_curve
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 9})
print('Imports OK')
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
try:
    import torch_geometric
    print(f'PyG {torch_geometric.__version__}')
except Exception as e:
    print(f'PyG import error: {e}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 — Load graph data and graph stats
# ─────────────────────────────────────────────────────────────────────────────
print('Loading graph data...')
data_train = torch.load(os.path.join(OUT_GRF, 'graph_data_train.pt'), weights_only=False)
data_val   = torch.load(os.path.join(OUT_GRF, 'graph_data_val.pt'),   weights_only=False)
data_test  = torch.load(os.path.join(OUT_GRF, 'graph_data_test.pt'),  weights_only=False)

with open(os.path.join(OUT_GRF, 'graph_stats.json')) as f:
    stats = json.load(f)

with open(os.path.join(OUT_PRE, 'feature_names.json')) as f:
    feat_info = json.load(f)
    FEATURE_NAMES = feat_info['feature_names']  # 37 edge feature names

NODE_FEAT_DIM  = stats['node_feat_dim']    # 11
EDGE_FEAT_DIM  = stats['edge_feat_dim']    # 37
N_NODES        = stats['n_nodes']          # 51
N_CLASSES_MULTI = stats['n_classes_multi'] # 10
CLASS_NAMES    = stats['class_names']      # ['Analysis', 'Backdoor', ...]
UNK_NODE_IDX   = stats['unk_node_idx']     # 50

print(f'\nGraph schema:')
print(f'  Nodes: {N_NODES} (known={stats["n_nodes_known"]}, UNK={UNK_NODE_IDX})')
print(f'  Node feat dim: {NODE_FEAT_DIM}, Edge feat dim: {EDGE_FEAT_DIM}')
print(f'  Classes: {N_CLASSES_MULTI} → {CLASS_NAMES}')

print(f'\nSplit edges:')
for name, d in [('train', data_train), ('val', data_val), ('test', data_test)]:
    attack_r = stats['splits'][name]['attack_edge_ratio']
    print(f'  {name:5s}: {d.num_edges:>9,} edges | attack={attack_r:.2%} | x={tuple(d.x.shape)} | edge_attr={tuple(d.edge_attr.shape)}')

print(f'\n⚠ Test distribution note: val attack={stats["splits"]["val"]["attack_edge_ratio"]:.2%}, test attack={stats["splits"]["test"]["attack_edge_ratio"]:.2%} — expect metric shift.')
print('⚠ Test set: all edges connect to UNK node (index 50) — edge features carry all discriminative signal on test.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Reproducibility, device, class weights
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    _vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {_vram_gb:.1f} GB')
    # GATv2Conv attention tensors are [E, heads, hd]:
    # (1.6M edges, 4 heads, 128 hd) = 3.3 GB each — 3 needed simultaneously
    # → OOM on ≤16 GB GPUs (T4, V100-16 GB) after prior model allocations.
    if _vram_gb < 20 and not REDUCE_HEADS:
        REDUCE_HEADS = True
        print(f'  ⚠ VRAM {_vram_gb:.1f} GB < 20 GB — REDUCE_HEADS auto-enabled '
              f'(GAT: 4→2 heads, FULL: 4→2 heads, halves attention memory)')
    elif REDUCE_HEADS:
        print('  ⚠ REDUCE_HEADS=True (manually set)')

# ── Binary class weight ──────────────────────────────────────────────────────
n_pos = (data_train.y == 1).sum().item()
n_neg = (data_train.y == 0).sum().item()
pos_weight_val = n_neg / max(n_pos, 1)
print(f'\nBinary: Normal={n_neg:,}, Attack={n_pos:,} → pos_weight={pos_weight_val:.2f}')
POS_WEIGHT = torch.tensor([pos_weight_val], dtype=torch.float32)

# ── Multiclass class weights (inverse frequency) ─────────────────────────────
counts_mc = torch.bincount(data_train.y_multi.long(), minlength=N_CLASSES_MULTI).float()
class_weights_mc = 1.0 / (counts_mc + 1e-8)
class_weights_mc = class_weights_mc / (class_weights_mc.sum() * N_CLASSES_MULTI)
print('Multiclass class weights (normalised inverse-freq):')
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights_mc.tolist())):
    print(f'  [{i}] {name:<16}: {w:.6f}  (count={int(counts_mc[i]):,})')

## Model Architectures

All four variants follow the same **edge classification** forward pass:
```
1. Message passing → node embeddings h [51, hidden_dim]
2. Edge repr = cat[h_src ‖ edge_feat_proj ‖ h_dst]  [E, repr_dim]
3. Edge MLP → logits [E, num_classes]
```

| Variant | Layer 1 → Layer 2 | Edge repr dim |
|---|---|---|
| GCN | GCNConv(11→hd) → GCNConv(hd→hd) | 2·hd + 37 |
| GAT | GATv2Conv(11→hd, h=4) → GATv2Conv(4·hd→hd, h=1) | 2·hd + 64 |
| SAGE | SAGEConv(11→hd) → SAGEConv(hd→hd) | 2·hd + 37 |
| FULL | GATv2Conv(11→128,h=4) → SAGEConv(512→256) | 576 |

Edge MLP shared tail: `Linear(repr_dim→256) + BN + ReLU + Dropout → Linear(256→128) + ReLU → Linear(128→K)`

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Model definitions
# ─────────────────────────────────────────────────────────────────────────────
def _build_edge_mlp(in_dim: int, num_classes: int, dropout: float) -> nn.Sequential:
    """Shared edge classifier MLP tail: in_dim → 256 → 128 → num_classes."""
    return nn.Sequential(
        nn.Linear(in_dim, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Linear(128, num_classes),
    )


class NIDS_GCN(nn.Module):
    """GCNConv × 2. Edge features only in the edge classifier head."""
    def __init__(self, node_feat_dim: int, edge_feat_dim: int,
                 hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.conv1   = GCNConv(node_feat_dim, hidden_dim, add_self_loops=True)
        self.conv2   = GCNConv(hidden_dim, hidden_dim, add_self_loops=True)
        self.drop    = nn.Dropout(dropout)
        self.edge_mlp = _build_edge_mlp(2 * hidden_dim + edge_feat_dim, num_classes, dropout)

    def forward(self, data: Data) -> torch.Tensor:
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        h = self.drop(F.relu(self.conv1(x, ei)))
        h = self.drop(F.relu(self.conv2(h, ei)))
        e = torch.cat([h[ei[0]], ea, h[ei[1]]], dim=-1)
        if self.training:  # recomputes MLP activations in backward → saves ~2.4 GB
            return grad_checkpoint(self.edge_mlp, e, use_reentrant=False)
        return self.edge_mlp(e)

    def get_node_embeddings(self, data: Data) -> torch.Tensor:
        x, ei = data.x, data.edge_index
        h = F.relu(self.conv1(x, ei))
        return F.relu(self.conv2(h, ei))


class NIDS_GAT(nn.Module):
    """GATv2Conv × 2 (4 heads → 1 head). Edge features in attention + classifier head."""
    def __init__(self, node_feat_dim: int, edge_feat_dim: int,
                 hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        heads1 = 2 if REDUCE_HEADS else 4
        self.edge_proj = nn.Sequential(nn.Linear(edge_feat_dim, 64), nn.ReLU())
        self.conv1 = GATv2Conv(node_feat_dim, hidden_dim, heads=heads1,
                               edge_dim=64, dropout=dropout, concat=True)
        self.conv2 = GATv2Conv(hidden_dim * heads1, hidden_dim, heads=1,
                               edge_dim=64, dropout=dropout, concat=False)
        self.drop  = nn.Dropout(dropout)
        self.edge_mlp = _build_edge_mlp(2 * hidden_dim + 64, num_classes, dropout)

    def forward(self, data: Data, return_attention: bool = False):
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        ep = self.edge_proj(ea)  # [E, 64]
        if return_attention:
            h1, att1 = self.conv1(x,  ei, edge_attr=ep, return_attention_weights=True)
            h1 = self.drop(F.relu(h1))
            h2, att2 = self.conv2(h1, ei, edge_attr=ep, return_attention_weights=True)
            h2 = self.drop(F.relu(h2))
            e = torch.cat([h2[ei[0]], ep, h2[ei[1]]], dim=-1)
            return self.edge_mlp(e), (att1, att2)
        if self.training:
            # Checkpoint each GAT conv: avoids storing [E, heads, hd] attention
            # tensors for backward (~4.8 GB total for 1.6M edges, 2 convs, 2 heads)
            def _c1(x_, ei_, ep_): return F.relu(self.conv1(x_, ei_, edge_attr=ep_))
            def _c2(h_, ei_, ep_): return F.relu(self.conv2(h_, ei_, edge_attr=ep_))
            h = self.drop(grad_checkpoint(_c1, x, ei, ep, use_reentrant=False))
            h = self.drop(grad_checkpoint(_c2, h, ei, ep, use_reentrant=False))
            e = torch.cat([h[ei[0]], ep, h[ei[1]]], dim=-1)
            return grad_checkpoint(self.edge_mlp, e, use_reentrant=False)
        h = self.drop(F.relu(self.conv1(x,  ei, edge_attr=ep)))
        h = self.drop(F.relu(self.conv2(h,  ei, edge_attr=ep)))
        e = torch.cat([h[ei[0]], ep, h[ei[1]]], dim=-1)
        return self.edge_mlp(e)

    def get_node_embeddings(self, data: Data) -> torch.Tensor:
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        ep = self.edge_proj(ea)
        h  = F.relu(self.conv1(x, ei, edge_attr=ep))
        return F.relu(self.conv2(h, ei, edge_attr=ep))


class NIDS_SAGE(nn.Module):
    """SAGEConv × 2 (mean aggregation). Edge features only in classifier head."""
    def __init__(self, node_feat_dim: int, edge_feat_dim: int,
                 hidden_dim: int, num_classes: int, dropout: float):
        super().__init__()
        self.conv1    = SAGEConv(node_feat_dim, hidden_dim, aggr='mean')
        self.conv2    = SAGEConv(hidden_dim, hidden_dim, aggr='mean')
        self.drop     = nn.Dropout(dropout)
        self.edge_mlp = _build_edge_mlp(2 * hidden_dim + edge_feat_dim, num_classes, dropout)

    def forward(self, data: Data) -> torch.Tensor:
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        h = self.drop(F.relu(self.conv1(x, ei)))
        h = self.drop(F.relu(self.conv2(h, ei)))
        e = torch.cat([h[ei[0]], ea, h[ei[1]]], dim=-1)
        if self.training:
            return grad_checkpoint(self.edge_mlp, e, use_reentrant=False)
        return self.edge_mlp(e)

    def get_node_embeddings(self, data: Data) -> torch.Tensor:
        x, ei = data.x, data.edge_index
        h = F.relu(self.conv1(x, ei))
        return F.relu(self.conv2(h, ei))


class NIDS_FULL(nn.Module):
    """GATv2Conv(4h) → SAGEConv → edge MLP. Edge features in attention + classifier."""
    def __init__(self, node_feat_dim: int, edge_feat_dim: int,
                 num_classes: int, dropout: float):
        super().__init__()
        heads = 2 if REDUCE_HEADS else 4
        gat_hidden = 64 if REDUCE_HEADS else 128
        self.edge_proj = nn.Sequential(nn.Linear(edge_feat_dim, 64), nn.ReLU())
        self.gat  = GATv2Conv(node_feat_dim, gat_hidden, heads=heads,
                              edge_dim=64, dropout=dropout, concat=True)
        sage_in   = gat_hidden * heads   # 512 (256 if REDUCE_HEADS)
        sage_out  = 256
        self.sage = SAGEConv(sage_in, sage_out, aggr='mean')
        self.drop = nn.Dropout(dropout)
        # repr dim: h_src(256) + e_proj(64) + h_dst(256) = 576
        self.edge_mlp = _build_edge_mlp(sage_out * 2 + 64, num_classes, dropout)

    def forward(self, data: Data) -> torch.Tensor:
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        ep = self.edge_proj(ea)                                         # [E, 64]
        if self.training:
            # Checkpoint GAT: avoids storing [E, heads, hd] attention intermediates
            def _gat(x_, ei_, ep_): return F.relu(self.gat(x_, ei_, edge_attr=ep_))
            h = self.drop(grad_checkpoint(_gat, x, ei, ep, use_reentrant=False))
        else:
            h = self.drop(F.relu(self.gat(x, ei, edge_attr=ep)))       # [N, 512]
        h = self.drop(F.relu(self.sage(h, ei)))                        # [N, 256]
        e = torch.cat([h[ei[0]], ep, h[ei[1]]], dim=-1)                # [E, 576]
        if self.training:
            return grad_checkpoint(self.edge_mlp, e, use_reentrant=False)
        return self.edge_mlp(e)

    def get_node_embeddings(self, data: Data) -> torch.Tensor:
        x, ei, ea = data.x, data.edge_index, data.edge_attr
        ep = self.edge_proj(ea)
        h  = F.relu(self.gat(x, ei, edge_attr=ep))
        return F.relu(self.sage(h, ei))


# Registry: maps name → (class, accepts_hidden_dim)
MODEL_REGISTRY = {
    'gcn':  (NIDS_GCN,  True),
    'gat':  (NIDS_GAT,  True),
    'sage': (NIDS_SAGE, True),
    'full': (NIDS_FULL, False),
}

def build_model(variant: str, num_classes: int, hidden_dim: int = 256, dropout: float = 0.3) -> nn.Module:
    """Instantiate a GNN variant by name."""
    cls, has_hd = MODEL_REGISTRY[variant]
    if has_hd:
        return cls(NODE_FEAT_DIM, EDGE_FEAT_DIM, hidden_dim, num_classes, dropout)
    return cls(NODE_FEAT_DIM, EDGE_FEAT_DIM, num_classes, dropout)

# Quick sanity check
for vname in ['gcn', 'gat', 'sage', 'full']:
    m = build_model(vname, num_classes=1)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)    
    print(f'{vname:4s}: {n_params:,} parameters')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9 — Training helpers: train_epoch + evaluate_gnn
# ─────────────────────────────────────────────────────────────────────────────
def train_epoch(model: nn.Module, data: Data, optimizer: torch.optim.Optimizer,
                criterion: nn.Module, task: str) -> float:
    """One full-batch training step. Returns scalar loss."""
    model.train()
    optimizer.zero_grad()
    out = model(data)              # [E, num_classes]
    if task == 'binary':
        loss = criterion(out.squeeze(-1), data.y.float())
    else:
        loss = criterion(out, data.y_multi.long())
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()
    return loss.item()


def evaluate_gnn(model: nn.Module, data: Data, task: str,
                 split_name: str = 'val', model_name: str = 'model'):
    """Full evaluation. Returns (metrics_dict, probs, labels, preds)."""
    model.eval()
    with torch.no_grad():
        out = model(data)      # [E, K]

    if task == 'binary':
        probs  = torch.sigmoid(out.squeeze(-1)).cpu().numpy()   # [E]
        preds  = (probs > 0.5).astype(int)
        labels = data.y.cpu().numpy()

        acc   = accuracy_score(labels, preds)
        prec  = precision_score(labels, preds, zero_division=0)
        rec   = recall_score(labels, preds, zero_division=0)
        f1    = f1_score(labels, preds, zero_division=0)
        fpr   = float(((preds == 1) & (labels == 0)).sum() / max((labels == 0).sum(), 1))
        mcc   = matthews_corrcoef(labels, preds)
        try:
            auc   = roc_auc_score(labels, probs)
        except Exception:
            auc   = float('nan')
        try:
            prauc = average_precision_score(labels, probs)
        except Exception:
            prauc = float('nan')

        metrics = {
            'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec),
            'f1': float(f1), 'fpr': float(fpr), 'roc_auc': float(auc),
            'pr_auc': float(prauc), 'mcc': float(mcc),
        }
        print(f'[{model_name}|{split_name}] Acc={acc:.4f} F1={f1:.4f} '
              f'AUC={auc:.4f} FPR={fpr:.4f} MCC={mcc:.4f} PR-AUC={prauc:.4f}')

    else:  # multiclass
        probs  = torch.softmax(out, dim=-1).cpu().numpy()  # [E, K]
        preds  = probs.argmax(axis=1)
        labels = data.y_multi.cpu().numpy()

        f1_mac = f1_score(labels, preds, average='macro',    zero_division=0)
        f1_wt  = f1_score(labels, preds, average='weighted', zero_division=0)
        acc    = accuracy_score(labels, preds)
        pcf1   = f1_score(labels, preds, average=None, zero_division=0, labels=list(range(N_CLASSES_MULTI)))
        metrics = {
            'f1_macro': float(f1_mac), 'f1_weighted': float(f1_wt), 'accuracy': float(acc),
            'per_class_f1': {c: float(f) for c, f in zip(CLASS_NAMES, pcf1)},
        }
        print(f'[{model_name}|{split_name}] Acc={acc:.4f} F1-macro={f1_mac:.4f} F1-wt={f1_wt:.4f}')
        per_str = ' '.join(f'{c[:3]}={f:.3f}' for c, f in zip(CLASS_NAMES, pcf1))
        print(f'  per-class: {per_str}')

    return metrics, probs, labels, preds


def _serialisable(obj):
    """Recursively convert numpy / torch types to JSON-serialisable Python types."""
    if isinstance(obj, dict):
        return {k: _serialisable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serialisable(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if hasattr(obj, 'item'):
        return obj.item()
    return obj

print('Helpers defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10 — train_gnn_full: full training loop with early stopping
# ─────────────────────────────────────────────────────────────────────────────
def train_gnn_full(
    variant: str,
    task: str,
    hidden_dim: int = 256,
    dropout: float = 0.3,
    lr: float = 1e-3,
    max_epochs: int = 100,
    patience: int = 10,
    hpo_mode: bool = False,
    hpo_epochs: int = 20,
    subsample_n: int = None,
    save_path: str = None,
    verbose: bool = True,
):
    """Train one GNN variant for one task. Returns (model, history_dict, best_val_metric).

    Args:
        variant:     'gcn' | 'gat' | 'sage' | 'full'
        task:        'binary' | 'multiclass'
        hpo_mode:    If True, stop at hpo_epochs regardless of early stopping.
        save_path:   If provided, save best state_dict here.
    """
    num_classes = 1 if task == 'binary' else N_CLASSES_MULTI
    epochs_to_run = hpo_epochs if hpo_mode else max_epochs

    # Build model
    model = build_model(variant, num_classes, hidden_dim, dropout).to(DEVICE)

    # Loss
    if task == 'binary':
        criterion = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights_mc.to(DEVICE))

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

    # ── Prepare training data (subsample edges for memory-safe HPO) ──────────
    if subsample_n is not None and data_train.num_edges > subsample_n:
        y_np   = data_train.y.numpy()
        idx0   = np.where(y_np == 0)[0]
        idx1   = np.where(y_np == 1)[0]
        k1     = max(1, int(subsample_n * len(idx1) / len(y_np)))
        k0     = subsample_n - k1
        rng_   = np.random.default_rng(SEED)
        chosen = np.concatenate([
            rng_.choice(idx0, min(k0, len(idx0)), replace=False),
            rng_.choice(idx1, min(k1, len(idx1)), replace=False),
        ])
        d_src = Data(
            x=data_train.x, edge_index=data_train.edge_index[:, chosen],
            edge_attr=data_train.edge_attr[chosen],
            y=data_train.y[chosen], y_multi=data_train.y_multi[chosen],
        )
        if verbose:
            print(f'  Subsample: {len(chosen):,}/{data_train.num_edges:,} edges '
                  f'(normal={min(k0,len(idx0)):,}, attack={min(k1,len(idx1)):,})')
    else:
        d_src = data_train

    # clone() → independent GPU copy; .to(DEVICE) will NOT mutate the global
    # Data object (PyG shallow-copies the _mapping dict, so without clone()
    # the global tensors silently move to GPU and are never freed)
    d_train = d_src.clone().to(DEVICE)
    d_val   = data_val.clone().to(DEVICE)

    best_val_metric = -1.0
    best_state      = None
    patience_ctr    = 0
    history = {'train_loss': [], 'val_metric': []}

    pbar = tqdm(range(1, epochs_to_run + 1),
                desc=f'{variant.upper()}-{task[:3]}',
                disable=not verbose)

    for epoch in pbar:
        train_loss = train_epoch(model, d_train, optimizer, criterion, task)
        scheduler.step()

        # Validate
        val_metrics, _, _, _ = evaluate_gnn(model, d_val, task,
                                            split_name='val',
                                            model_name=f'{variant}-{task[:3]}')
        val_m = val_metrics['f1'] if task == 'binary' else val_metrics['f1_macro']

        history['train_loss'].append(train_loss)
        history['val_metric'].append(val_m)

        pbar.set_postfix({'loss': f'{train_loss:.4f}', 'val_F1': f'{val_m:.4f}'})

        if val_m > best_val_metric:
            best_val_metric = val_m
            best_state      = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            patience_ctr    = 0
        else:
            patience_ctr += 1
            if not hpo_mode and patience_ctr >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch} (patience={patience})')
                break

    # Restore best weights
    model.load_state_dict(best_state)

    # Save checkpoint
    if save_path is not None:
        torch.save(best_state, save_path)
        if verbose:
            print(f'  Saved → {save_path}')

    # ── GPU memory cleanup ──────────────────────────────────────────────────
    # NOTE: .to('cpu') returns a new object — use del to actually release
    # GPU-resident tensors so the caching allocator can reclaim the memory.
    del d_train, d_val
    del optimizer, scheduler, criterion
    model.cpu()  # move returned model off GPU; callers do .to(DEVICE) before eval
    gc.collect()
    torch.cuda.empty_cache()

    return model, history, best_val_metric

print('train_gnn_full defined.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11 — HPO grid search (binary task, 20 epochs per config)
# ─────────────────────────────────────────────────────────────────────────────
HPO_GRID = {
    'hidden_dim': [128, 256],
    'dropout':    [0.3, 0.4],
    'lr':         [1e-3, 5e-4],
}
HPO_EPOCHS = 20
# Subsampling for HPO: 200k edges → ~2 GB peak (vs ~13 GB with full 1.6M edges)
# Full dataset is used during final training (Cells 13 & 17)
HPO_SUBSAMPLE_N = 200_000

hpo_results = {}   # {variant: {'best_config': {...}, 'best_val_f1': float}}
best_configs = {}

for variant in ['gcn', 'gat', 'sage', 'full']:
    print(f'\n── HPO: {variant.upper()} ─────────────────────────────────────')
    best_f1  = -1.0
    best_cfg = {}

    # hidden_dim search space per variant:
    #  FULL : fixed architecture — only dropout & lr vary
    #  GAT  : cap at 128 — attention tensors are [E, heads, hd];
    #         (1.6M, 4, 128) = 3.3 GB each → use max hd=128 to stay safe
    #  GCN/SAGE: full grid [128, 256]
    if variant == 'full':
        hd_list = [128]
    elif variant == 'gat':
        hd_list = [64, 128]
    else:
        hd_list = HPO_GRID['hidden_dim']

    for hd in hd_list:
        for do in HPO_GRID['dropout']:
            for lr in HPO_GRID['lr']:
                torch.manual_seed(SEED)
                _m, _, val_f1 = train_gnn_full(
                    variant, task='binary',
                    hidden_dim=hd, dropout=do, lr=lr,
                    hpo_mode=True, hpo_epochs=HPO_EPOCHS,
                    subsample_n=HPO_SUBSAMPLE_N,
                    verbose=False,
                )
                del _m          # model returned on CPU; free immediately
                gc.collect()    # force GC before next config
                torch.cuda.empty_cache()
                cfg_str = f'hd={hd}, do={do}, lr={lr}'
                print(f'  {cfg_str:<35} → val F1={val_f1:.4f}')
                if val_f1 > best_f1:
                    best_f1  = val_f1
                    best_cfg = {'hidden_dim': hd, 'dropout': do, 'lr': lr}

    hpo_results[variant] = {'best_config': best_cfg, 'best_val_f1': float(best_f1)}
    best_configs[variant] = best_cfg
    print(f'  → Best: {best_cfg}  val F1={best_f1:.4f}')
    gc.collect()          # inter-variant cleanup
    torch.cuda.empty_cache()

print('\n=== HPO Summary ===')
for v, r in hpo_results.items():
    print(f'  {v:4s}: {r["best_config"]}  → val F1={r["best_val_f1"]:.4f}')

## Phase 5.1 — Binary Classification Training

Each GNN variant is trained for up to 100 epochs with early stopping (patience=10) on validation F1.  
Loss: `BCEWithLogitsLoss(pos_weight=n_neg/n_pos)` to correct for class imbalance (attack=4.84% on train/val).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 13 — Train all 4 variants: binary task
# ─────────────────────────────────────────────────────────────────────────────
trained_models_bin = {}
histories_bin      = {}

for variant in ['gcn', 'gat', 'sage', 'full']:
    cfg  = best_configs[variant]
    path = os.path.join(OUT_DIR, f'nids_{variant}_binary.pt')
    print(f'\n═══ Training NIDS-{variant.upper()} | binary | config={cfg} ═══')
    t0 = time.time()
    model, hist, best_f1 = train_gnn_full(
        variant, task='binary',
        hidden_dim=cfg['hidden_dim'], dropout=cfg['dropout'], lr=cfg['lr'],
        max_epochs=100, patience=10,
        save_path=path, verbose=True,
    )
    elapsed = time.time() - t0
    trained_models_bin[variant] = model
    histories_bin[variant]      = hist
    print(f'  Done in {elapsed:.1f}s | best val F1={best_f1:.4f}')

print('\nAll binary models trained.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 14 — Evaluate binary: val + test for all 4 variants
# ─────────────────────────────────────────────────────────────────────────────
gnn_results = {}  # matches baseline_results.json schema: {name: {val:{...}, test:{...}}}
roc_data_bin = {}  # for plotting

d_val_gpu  = data_val.clone().to(DEVICE)
d_test_gpu = data_test.clone().to(DEVICE)

print('─' * 70)
print('BINARY CLASSIFICATION RESULTS')
print('─' * 70)

for variant, model in trained_models_bin.items():
    model = model.to(DEVICE)
    key   = f'nids_{variant}_binary'

    val_m,  val_p,  val_l,  _  = evaluate_gnn(model, d_val_gpu,  'binary', 'val',  f'NIDS-{variant.upper()}')
    test_m, test_p, test_l, _  = evaluate_gnn(model, d_test_gpu, 'binary', 'test', f'NIDS-{variant.upper()}')

    gnn_results[key] = {
        'val':  val_m,
        'test': test_m,
        'best_config': best_configs[variant],
    }

    # Store ROC curve data
    fpr_arr, tpr_arr, _ = roc_curve(test_l, test_p)
    roc_data_bin[f'NIDS-{variant.upper()}'] = (fpr_arr, tpr_arr, test_m['roc_auc'])

    model.to('cpu')

d_val_gpu.to('cpu'); d_test_gpu.to('cpu')
torch.cuda.empty_cache()

print('\n=== Binary Summary (Test) ===')
print(f'{"Model":<18} {"Acc":>6} {"Prec":>6} {"Rec":>6} {"F1":>6} {"AUC":>6} {"FPR":>6} {"MCC":>6}')
for k, v in gnn_results.items():
    if 'binary' in k:
        t = v['test']
        n = k.replace('nids_','NIDS-').replace('_binary','')
        print(f'{n:<18} {t["accuracy"]:>6.4f} {t["precision"]:>6.4f} '
              f'{t["recall"]:>6.4f} {t["f1"]:>6.4f} {t["roc_auc"]:>6.4f} '
              f'{t["fpr"]:>6.4f} {t["mcc"]:>6.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 15 — Binary training curves: loss + val F1 for all 4 variants
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = {'gcn': '#2196F3', 'gat': '#FF5722', 'sage': '#4CAF50', 'full': '#9C27B0'}

for variant, hist in histories_bin.items():
    ep = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(ep, hist['train_loss'],  color=colors[variant], label=f'NIDS-{variant.upper()}')
    axes[1].plot(ep, hist['val_metric'], color=colors[variant], label=f'NIDS-{variant.upper()}')

axes[0].set(title='Training Loss (Binary)', xlabel='Epoch', ylabel='BCE Loss')
axes[1].set(title='Validation F1 (Binary)', xlabel='Epoch', ylabel='F1 Score')
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'gnn_training_curves_binary.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')

## Phase 5.2 — Multiclass Classification Training

10-class edge classification: `{Analysis, Backdoor, DoS, Exploits, Fuzzers, Generic, Normal, Reconnaissance, Shellcode, Worms}`  
Loss: `CrossEntropyLoss(weight=inverse_freq)`.  Early stopping on val **F1-macro**.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 17 — Train all 4 variants: multiclass task
# ─────────────────────────────────────────────────────────────────────────────
trained_models_mc = {}
histories_mc      = {}

for variant in ['gcn', 'gat', 'sage', 'full']:
    cfg  = best_configs[variant]
    path = os.path.join(OUT_DIR, f'nids_{variant}_multiclass.pt')
    print(f'\n═══ Training NIDS-{variant.upper()} | multiclass | config={cfg} ═══')
    t0 = time.time()
    model, hist, best_f1 = train_gnn_full(
        variant, task='multiclass',
        hidden_dim=cfg['hidden_dim'], dropout=cfg['dropout'], lr=cfg['lr'],
        max_epochs=100, patience=10,
        save_path=path, verbose=True,
    )
    elapsed = time.time() - t0
    trained_models_mc[variant] = model
    histories_mc[variant]      = hist
    print(f'  Done in {elapsed:.1f}s | best val F1-macro={best_f1:.4f}')

print('\nAll multiclass models trained.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 18 — Evaluate multiclass: val + test for all 4 variants
# ─────────────────────────────────────────────────────────────────────────────
d_val_gpu  = data_val.clone().to(DEVICE)
d_test_gpu = data_test.clone().to(DEVICE)

print('─' * 70)
print('MULTICLASS CLASSIFICATION RESULTS')
print('─' * 70)

for variant, model in trained_models_mc.items():
    model = model.to(DEVICE)
    key   = f'nids_{variant}_multiclass'

    val_m,  _, _, _ = evaluate_gnn(model, d_val_gpu,  'multiclass', 'val',  f'NIDS-{variant.upper()}')
    test_m, _, _, _ = evaluate_gnn(model, d_test_gpu, 'multiclass', 'test', f'NIDS-{variant.upper()}')

    gnn_results[key] = {
        'val':  val_m,
        'test': test_m,
        'best_config': best_configs[variant],
    }
    model.to('cpu')

d_val_gpu.to('cpu'); d_test_gpu.to('cpu')
torch.cuda.empty_cache()

print('\n=== Multiclass Summary (Test) ===')
print(f'{"Model":<22} {"Acc":>6} {"F1-mac":>7} {"F1-wt":>7}')
for k, v in gnn_results.items():
    if 'multiclass' in k:
        t = v['test']
        n = k.replace('nids_','NIDS-').replace('_multiclass','')
        print(f'{n:<22} {t["accuracy"]:>6.4f} {t["f1_macro"]:>7.4f} {t["f1_weighted"]:>7.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 19 — Multiclass training curves
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for variant, hist in histories_mc.items():
    ep = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(ep, hist['train_loss'],  color=colors[variant], label=f'NIDS-{variant.upper()}')
    axes[1].plot(ep, hist['val_metric'], color=colors[variant], label=f'NIDS-{variant.upper()}')

axes[0].set(title='Training Loss (Multiclass)', xlabel='Epoch', ylabel='CE Loss')
axes[1].set(title='Validation F1-macro (Multiclass)', xlabel='Epoch', ylabel='F1-macro')
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'gnn_training_curves_multiclass.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 20 — ROC curves: all 4 GNN binary models + RF baseline
# ─────────────────────────────────────────────────────────────────────────────
# Load baseline ROC data from saved results (Phase 4 artifact — read-only input)
with open(os.path.join(OUT_MDL_IN, 'baseline_results.json')) as f:
    baseline_results = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Test ROC ──────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')

# GNN ROC curves
gnn_colors_plot = {'GCN': '#2196F3', 'GAT': '#FF5722', 'SAGE': '#4CAF50', 'FULL': '#9C27B0'}
for variant, model in trained_models_bin.items():
    model = model.to(DEVICE)
    model.eval()
    d_test_gpu = data_test.clone().to(DEVICE)
    with torch.no_grad():
        out = model(d_test_gpu)
    probs = torch.sigmoid(out.squeeze(-1)).cpu().numpy()
    labels = d_test_gpu.y.cpu().numpy()
    fpr_r, tpr_r, _ = roc_curve(labels, probs)
    auc_v = roc_auc_score(labels, probs)
    ax.plot(fpr_r, tpr_r, color=gnn_colors_plot[variant.upper()],
            label=f'NIDS-{variant.upper()} (AUC={auc_v:.3f})', linewidth=1.8)
    model.to('cpu')

# RF baseline (annotate from saved metrics)
rf_auc = baseline_results['rf_binary']['test']['roc_auc']
ax.annotate(f'RF baseline AUC={rf_auc:.3f}', xy=(0.6, 0.35), fontsize=8,
            color='gray', style='italic')

ax.set(title='ROC Curves — Binary (Test Set, attack=55.06%)',
       xlabel='False Positive Rate', ylabel='True Positive Rate')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

# ── Val ROC ───────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
for variant, model in trained_models_bin.items():
    model = model.to(DEVICE)
    model.eval()
    d_val_gpu = data_val.clone().to(DEVICE)
    with torch.no_grad():
        out = model(d_val_gpu)
    probs  = torch.sigmoid(out.squeeze(-1)).cpu().numpy()
    labels = d_val_gpu.y.cpu().numpy()
    fpr_r, tpr_r, _ = roc_curve(labels, probs)
    auc_v = roc_auc_score(labels, probs)
    ax.plot(fpr_r, tpr_r, color=gnn_colors_plot[variant.upper()],
            label=f'NIDS-{variant.upper()} (AUC={auc_v:.3f})', linewidth=1.8)
    model.to('cpu')

ax.set(title='ROC Curves — Binary (Val Set, attack=4.84%)',
       xlabel='False Positive Rate', ylabel='True Positive Rate')
ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'gnn_roc_curves.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')
d_val_gpu.to('cpu'); d_test_gpu.to('cpu'); torch.cuda.empty_cache()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 21 — Precision-Recall curves (binary, test set)
# ─────────────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

d_test_gpu = data_test.clone().to(DEVICE)
for variant, model in trained_models_bin.items():
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model(d_test_gpu)
    probs  = torch.sigmoid(out.squeeze(-1)).cpu().numpy()
    labels = d_test_gpu.y.cpu().numpy()
    prec_arr, rec_arr, _ = precision_recall_curve(labels, probs)
    pr_auc = average_precision_score(labels, probs)
    ax.plot(rec_arr, prec_arr, color=gnn_colors_plot[variant.upper()],
            label=f'NIDS-{variant.upper()} (AP={pr_auc:.3f})', linewidth=1.8)
    model.to('cpu')

# RF baseline AP
rf_prauc = baseline_results['rf_binary']['test'].get('pr_auc', 'N/A')
baseline_attack_ratio = 0.5506
ax.axhline(baseline_attack_ratio, linestyle='--', color='gray', alpha=0.5,
           label=f'Random classifier baseline ({baseline_attack_ratio:.2f})')

ax.set(title='Precision–Recall Curves — Binary (Test Set)',
       xlabel='Recall', ylabel='Precision', xlim=[0, 1], ylim=[0, 1])
ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'gnn_pr_curves.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')
d_test_gpu.to('cpu'); torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 22 — F1-macro comparison bar chart: GNNs vs all baselines
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Binary F1 comparison ──────────────────────────────────────────────────────
baseline_models = ['lr_binary', 'rf_binary', 'xgb_binary', 'lgbm_binary', 'mlp_binary']
baseline_labels = ['LR', 'RF', 'XGBoost', 'LightGBM', 'MLP']

val_f1_all  = [baseline_results[m]['val']['f1']  for m in baseline_models]
test_f1_all = [baseline_results[m]['test']['f1'] for m in baseline_models]

for variant in ['gcn', 'gat', 'sage', 'full']:
    k = f'nids_{variant}_binary'
    val_f1_all.append(gnn_results[k]['val']['f1'])
    test_f1_all.append(gnn_results[k]['test']['f1'])
    baseline_labels.append(f'NIDS-{variant.upper()}')

x  = np.arange(len(baseline_labels))
bw = 0.35
bar_colors = ['#90A4AE'] * 5 + [colors[v] for v in ['gcn', 'gat', 'sage', 'full']]
axes[0].bar(x - bw/2, val_f1_all,  bw, label='Val',  color=bar_colors, alpha=0.7)
axes[0].bar(x + bw/2, test_f1_all, bw, label='Test', color=bar_colors, alpha=1.0)
axes[0].set(title='Binary F1 — All Models', xlabel='', ylabel='F1 Score',
            xticks=x, xticklabels=baseline_labels)
axes[0].set_xticklabels(baseline_labels, rotation=35, ha='right')
axes[0].axhline(max(baseline_results[m]['test']['f1'] for m in baseline_models),
                linestyle='--', color='red', alpha=0.6, label='Best baseline')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# ── Multiclass F1-macro comparison ───────────────────────────────────────────
baseline_mc = ['lr_multiclass', 'rf_multiclass', 'xgb_multiclass', 'lgbm_multiclass', 'mlp_multiclass']
mc_labels   = ['LR', 'RF', 'XGBoost', 'LightGBM', 'MLP']

val_mac_all  = [baseline_results[m]['val']['f1_macro']  for m in baseline_mc]
test_mac_all = [baseline_results[m]['test']['f1_macro'] for m in baseline_mc]

for variant in ['gcn', 'gat', 'sage', 'full']:
    k = f'nids_{variant}_multiclass'
    val_mac_all.append(gnn_results[k]['val']['f1_macro'])
    test_mac_all.append(gnn_results[k]['test']['f1_macro'])
    mc_labels.append(f'NIDS-{variant.upper()}')

x2 = np.arange(len(mc_labels))
axes[1].bar(x2 - bw/2, val_mac_all,  bw, label='Val',  color=bar_colors, alpha=0.7)
axes[1].bar(x2 + bw/2, test_mac_all, bw, label='Test', color=bar_colors, alpha=1.0)
axes[1].set(title='Multiclass F1-macro — All Models', xlabel='', ylabel='F1-macro',
            xticks=x2, xticklabels=mc_labels)
axes[1].set_xticklabels(mc_labels, rotation=35, ha='right')
axes[1].axhline(max(baseline_results[m]['test']['f1_macro'] for m in baseline_mc),
                linestyle='--', color='red', alpha=0.6, label='Best baseline')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'gnn_vs_baseline.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 23 — Confusion matrix: best binary GNN on test set
# ─────────────────────────────────────────────────────────────────────────────
# Identify best binary GNN by test F1
best_bin_variant = max(
    ['gcn', 'gat', 'sage', 'full'],
    key=lambda v: gnn_results[f'nids_{v}_binary']['test']['f1']
)
print(f'Best binary GNN: NIDS-{best_bin_variant.upper()} '
      f'(test F1={gnn_results[f"nids_{best_bin_variant}_binary"]["test"]["f1"]:.4f})')

model_b = trained_models_bin[best_bin_variant].to(DEVICE)
model_b.eval()
d_test_gpu = data_test.clone().to(DEVICE)
with torch.no_grad():
    out = model_b(d_test_gpu)
probs_b = torch.sigmoid(out.squeeze(-1)).cpu().numpy()
preds_b = (probs_b > 0.5).astype(int)
labels_b = d_test_gpu.y.cpu().numpy()
model_b.to('cpu'); d_test_gpu.to('cpu'); torch.cuda.empty_cache()

cm_b = confusion_matrix(labels_b, preds_b)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, norm in zip(axes, [None, 'true']):
    cm_plot = confusion_matrix(labels_b, preds_b, normalize=norm)
    fmt = '.2%' if norm else 'd'
    sns.heatmap(cm_plot, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
    ax.set(xlabel='Predicted', ylabel='True',
           title=f'NIDS-{best_bin_variant.upper()} Binary CM (Test) — '
                 + ('Counts' if norm is None else 'Normalised'))

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'cm_best_gnn_binary.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 24 — Confusion matrix + per-class F1: best multiclass GNN on test set
# ─────────────────────────────────────────────────────────────────────────────
best_mc_variant = max(
    ['gcn', 'gat', 'sage', 'full'],
    key=lambda v: gnn_results[f'nids_{v}_multiclass']['test']['f1_macro']
)
print(f'Best multiclass GNN: NIDS-{best_mc_variant.upper()} '
      f'(test F1-macro={gnn_results[f"nids_{best_mc_variant}_multiclass"]["test"]["f1_macro"]:.4f})')

model_mc = trained_models_mc[best_mc_variant].to(DEVICE)
model_mc.eval()
d_test_gpu = data_test.clone().to(DEVICE)
with torch.no_grad():
    out_mc = model_mc(d_test_gpu)
preds_mc  = out_mc.argmax(dim=1).cpu().numpy()
labels_mc = d_test_gpu.y_multi.cpu().numpy()
model_mc.to('cpu'); d_test_gpu.to('cpu'); torch.cuda.empty_cache()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Confusion matrix
cm_mc = confusion_matrix(labels_mc, preds_mc, normalize='true')
sns.heatmap(cm_mc, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0],
            xticklabels=[c[:5] for c in CLASS_NAMES],
            yticklabels=[c[:5] for c in CLASS_NAMES])
axes[0].set(xlabel='Predicted', ylabel='True',
            title=f'NIDS-{best_mc_variant.upper()} Multiclass CM (Test, row-normalised)')

# Per-class F1 bar chart — GNNs vs best baseline (LightGBM)
lgbm_pcf1 = baseline_results['lgbm_multiclass']['test']['per_class_f1']
gnn_pcf1  = gnn_results[f'nids_{best_mc_variant}_multiclass']['test']['per_class_f1']
x3 = np.arange(len(CLASS_NAMES))
bw2 = 0.35
axes[1].bar(x3 - bw2/2, [lgbm_pcf1.get(c, 0) for c in CLASS_NAMES],
            bw2, label='LightGBM (best baseline)', color='#90A4AE', alpha=0.8)
axes[1].bar(x3 + bw2/2, [gnn_pcf1.get(c, 0) for c in CLASS_NAMES],
            bw2, label=f'NIDS-{best_mc_variant.upper()}', color=colors[best_mc_variant], alpha=0.9)
axes[1].set(title='Per-class F1 — Best Multiclass GNN vs LightGBM (Test)',
            xlabel='Class', ylabel='F1', xticks=x3, xticklabels=CLASS_NAMES)
axes[1].set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_p = os.path.join(OUT_DIR, 'cm_best_gnn_multiclass.png')
plt.savefig(save_p, bbox_inches='tight'); plt.show()
print(f'Saved → {save_p}')

save_p2 = os.path.join(OUT_DIR, 'gnn_perclass_f1.png')
fig2, ax2 = plt.subplots(figsize=(12, 4))
ax2.bar(x3 - bw2/2, [lgbm_pcf1.get(c, 0) for c in CLASS_NAMES], bw2,
        label='LightGBM', color='#90A4AE', alpha=0.8)
ax2.bar(x3 + bw2/2, [gnn_pcf1.get(c, 0) for c in CLASS_NAMES], bw2,
        label=f'NIDS-{best_mc_variant.upper()}', color=colors[best_mc_variant], alpha=0.9)
ax2.set(title='Per-class F1 (Test)', xlabel='Class', ylabel='F1', xticks=x3)
ax2.set_xticklabels(CLASS_NAMES, rotation=35, ha='right')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(save_p2, bbox_inches='tight')
print(f'Saved → {save_p2}')

---
## Phase 5.3 — Comprehensive GNN Evaluation Suite

Beyond standard classification metrics, the following evaluations are applied:

| # | Evaluation | What it measures |
|---|---|---|
| A | **Calibration** | Are predicted probabilities reliable? ECE, Brier Score, reliability diagram |
| B | **MC Dropout Uncertainty** | Epistemic uncertainty via 30 stochastic forward passes |
| C | **Node Embedding t-SNE** | Are malicious IPs separable in embedding space? |
| D | **Edge Embedding UMAP** | Are attack edges separable in edge repr space? |
| E | **GAT Attention Analysis** | Which edges get highest attention per attack class? |
| F | **Ablation Studies** | Impact of removing edge features / node features / layers |
| G | **Robustness** | Degradation under DropEdge, feature noise, by degree bucket |
| H | **Gradient Attribution** | Which edge features most influence attack predictions? |

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 26 — A: Calibration — Reliability diagram, ECE, Brier Score
    # ─────────────────────────────────────────────────────────────────────────────
    def expected_calibration_error(y_true, y_prob, n_bins=10):
        """Compute Expected Calibration Error (ECE)."""
        bin_edges  = np.linspace(0.0, 1.0, n_bins + 1)
        ece        = 0.0
        n          = len(y_true)
        for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
            mask   = (y_prob >= lo) & (y_prob < hi)
            if mask.sum() == 0:
                continue
            acc_bin  = y_true[mask].mean()
            conf_bin = y_prob[mask].mean()
            ece     += mask.sum() / n * abs(acc_bin - conf_bin)
        return float(ece)
    
    eval_suite = {'calibration': {}, 'uncertainty': {}, 'ablation': {}, 'robustness': {}}
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    d_val_gpu  = data_val.clone().to(DEVICE)
    d_test_gpu = data_test.clone().to(DEVICE)
    
    for idx, (variant, model) in enumerate(trained_models_bin.items()):
        model = model.to(DEVICE)
        model.eval()
    
        calib = {}
        for split_name, d_gpu in [('val', d_val_gpu), ('test', d_test_gpu)]:
            with torch.no_grad():
                out = model(d_gpu)
            probs  = torch.sigmoid(out.squeeze(-1)).cpu().numpy()
            labels = d_gpu.y.cpu().numpy()
    
            ece    = expected_calibration_error(labels, probs)
            brier  = brier_score_loss(labels, probs)
            calib[split_name] = {'ece': float(ece), 'brier': float(brier)}
            print(f'[{variant.upper()}|{split_name}] ECE={ece:.4f}  Brier={brier:.4f}')
    
            # Reliability diagram (test only)
            if split_name == 'test':
                fraction_pos, mean_pred_val = calibration_curve(labels, probs, n_bins=10)
                ax = axes[idx]
                ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
                ax.plot(mean_pred_val, fraction_pos, 'o-',
                        color=colors[variant], label=f'ECE={ece:.3f}')
                ax.set(title=f'NIDS-{variant.upper()} (Test)',
                       xlabel='Mean predicted probability', ylabel='Fraction of positives')
                ax.legend(fontsize=7); ax.grid(alpha=0.3)
    
        eval_suite['calibration'][variant] = calib
        model.to('cpu')
    
    d_val_gpu.to('cpu'); d_test_gpu.to('cpu'); torch.cuda.empty_cache()
    
    plt.suptitle('Reliability Diagrams — Binary GNN Models (Test Set)', y=1.01)
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_calibration.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 27 — B: MC Dropout Uncertainty Estimation
    # Epistemic uncertainty via T=30 stochastic forward passes (dropout active at inference)
    # ─────────────────────────────────────────────────────────────────────────────
    T_MC = 30  # number of MC samples
    
    def mc_dropout_predict(model: nn.Module, data: Data, T: int, task: str):
        """T stochastic forward passes with dropout active. Returns (mean_probs, std_probs)."""
        model.train()  # enable dropout
        all_probs = []
        with torch.no_grad():
            for _ in range(T):
                out = model(data)
                if task == 'binary':
                    all_probs.append(torch.sigmoid(out.squeeze(-1)).cpu().numpy())
                else:
                    all_probs.append(torch.softmax(out, dim=-1).cpu().numpy())
        model.eval()
        all_probs = np.stack(all_probs, axis=0)  # [T, E] or [T, E, K]
        return all_probs.mean(axis=0), all_probs.std(axis=0)
    
    # Use best binary model on val set (manageable size: 411k edges)
    model_unc = trained_models_bin[best_bin_variant].to(DEVICE)
    d_val_gpu  = data_val.clone().to(DEVICE)
    
    print(f'Running {T_MC} MC-Dropout passes on val set ({data_val.num_edges:,} edges)...')
    mean_p, std_p = mc_dropout_predict(model_unc, d_val_gpu, T=T_MC, task='binary')
    labels_v = data_val.y.numpy()
    preds_mc_det = (mean_p > 0.5).astype(int)
    
    correct_mask = (preds_mc_det == labels_v)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Uncertainty distribution: correct vs wrong
    axes[0].hist(std_p[correct_mask],   bins=50, alpha=0.6, label='Correct',    color='#4CAF50', density=True)
    axes[0].hist(std_p[~correct_mask],  bins=50, alpha=0.6, label='Incorrect',  color='#F44336', density=True)
    axes[0].set(title=f'Uncertainty (std) Distribution — NIDS-{best_bin_variant.upper()}',
                xlabel='Std of MC predictions', ylabel='Density')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    
    # Uncertainty by true label
    axes[1].boxplot([std_p[labels_v == 0], std_p[labels_v == 1]],
                   labels=['Normal', 'Attack'], patch_artist=True,
                   boxprops=dict(facecolor='#90A4AE'))
    axes[1].set(title='Uncertainty by True Label', ylabel='Prediction Std')
    axes[1].grid(alpha=0.3)
    
    # Calibration of MC mean vs deterministic predictions
    det_eval, _, _, _ = evaluate_gnn(model_unc, d_val_gpu, 'binary', 'val', f'MC-{best_bin_variant}')
    mc_f1   = f1_score(labels_v, preds_mc_det, zero_division=0)
    axes[2].bar(['Deterministic\n(1 pass)', f'MC-Dropout\n({T_MC} passes)'],
                [det_eval['f1'], mc_f1], color=['#90A4AE', '#9C27B0'], alpha=0.9)
    axes[2].set(title='Deterministic vs MC-Dropout F1 (Val)', ylabel='F1 Score', ylim=[0, 1])
    axes[2].grid(axis='y', alpha=0.3)
    
    eval_suite['uncertainty'][best_bin_variant] = {
        'mean_std_correct':   float(std_p[correct_mask].mean()),
        'mean_std_incorrect': float(std_p[~correct_mask].mean()),
        'mean_std_normal':    float(std_p[labels_v == 0].mean()),
        'mean_std_attack':    float(std_p[labels_v == 1].mean()),
        'mc_f1':              float(mc_f1),
        'det_f1':             float(det_eval['f1']),
    }
    print(f'Mean uncertainty — correct={std_p[correct_mask].mean():.4f}, incorrect={std_p[~correct_mask].mean():.4f}')
    
    model_unc.to('cpu'); d_val_gpu.to('cpu'); torch.cuda.empty_cache()
    
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_uncertainty.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 28 — C: Node Embedding t-SNE
    # Visualise 51-node embeddings from final GNN layer, coloured by attack ratio
    # ─────────────────────────────────────────────────────────────────────────────
    from sklearn.manifold import TSNE
    
    # Load node metadata
    node_df = pd.read_csv(os.path.join(OUT_PRE, 'node_features.csv'))
    with open(os.path.join(OUT_PRE, 'node_features_meta.json')) as f:
        node_meta = json.load(f)
    node_feat_cols = node_meta['node_feat_cols']
    
    # Extract embeddings from best binary model
    model_emb = trained_models_bin[best_bin_variant].to(DEVICE)
    model_emb.eval()
    d_train_gpu = data_train.clone().to(DEVICE)
    
    with torch.no_grad():
        node_emb = model_emb.get_node_embeddings(d_train_gpu)  # [51, hidden_dim]
    
    node_emb_np = node_emb.cpu().numpy()   # [51, hd]
    model_emb.to('cpu'); d_train_gpu.to('cpu'); torch.cuda.empty_cache()
    
    # t-SNE on 51 nodes (trivially fast)
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, N_NODES - 1))
    emb_2d = tsne.fit_transform(node_emb_np)   # [51, 2]
    
    # Attack ratio per node (from node_features.csv)
    if 'src_attack_ratio' in node_df.columns:
        attack_ratios = node_df['src_attack_ratio'].fillna(0).values
        # Append UNK node (index 50) with ratio=0
        attack_ratios = np.append(attack_ratios, [0.0])
    else:
        attack_ratios = np.zeros(N_NODES)
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # Scatter coloured by attack ratio
    sc = axes[0].scatter(emb_2d[:, 0], emb_2d[:, 1],
                         c=attack_ratios, cmap='RdYlGn_r', s=80, edgecolors='k', linewidths=0.5)
    plt.colorbar(sc, ax=axes[0], label='src_attack_ratio')
    # Mark UNK node
    axes[0].scatter(emb_2d[UNK_NODE_IDX, 0], emb_2d[UNK_NODE_IDX, 1],
                    marker='*', s=300, c='blue', zorder=5, label='UNK node')
    axes[0].set(title=f'Node Embeddings t-SNE — NIDS-{best_bin_variant.upper()} (51 nodes)',
                xlabel='t-SNE 1', ylabel='t-SNE 2')
    axes[0].legend()
    
    # Bar chart of attack ratio per node (top 20)
    sorted_idx = np.argsort(attack_ratios[:-1])[::-1][:20]  # top 20 (exclude UNK)
    if 'ip' in node_df.columns:
        ip_labels = node_df['ip'].astype(str).values[sorted_idx]
    else:
        ip_labels = [f'Node {i}' for i in sorted_idx]
    axes[1].barh(range(20), attack_ratios[sorted_idx], color='#F44336', alpha=0.8)
    axes[1].set_yticks(range(20))
    axes[1].set_yticklabels(ip_labels, fontsize=7)
    axes[1].set(title='Top 20 Nodes by Source Attack Ratio',
                xlabel='src_attack_ratio')
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_node_embeddings_tsne.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 29 — D: Edge Embedding UMAP/t-SNE
    # Visualise edge representations [E_val, repr_dim] — subsample 10k
    # ─────────────────────────────────────────────────────────────────────────────
    EDGE_VIZ_N = 10_000  # subsample size
    
    def extract_edge_repr(model: nn.Module, data: Data, device) -> np.ndarray:
        """Extract edge representations from the layer before the final classifier."""
        model.eval()
        hook_output = []
    
        def hook_fn(module, input, output):
            hook_output.append(output.detach().cpu())
    
        # Hook on the first Linear layer of edge_mlp (captures input repr)
        handle = model.edge_mlp[0].register_forward_hook(hook_fn)
        with torch.no_grad():
            model(data)
        handle.remove()
    
        return hook_output[0].numpy() if hook_output else np.array([])
    
    # Get edge reprs from best binary model on val set
    model_edg = trained_models_bin[best_bin_variant].to(DEVICE)
    d_val_gpu  = data_val.clone().to(DEVICE)
    
    print('Extracting edge representations via forward hook...')
    e_repr = extract_edge_repr(model_edg, d_val_gpu, DEVICE)  # [E_val, repr_dim]
    bin_labels_v = data_val.y.numpy()
    mc_labels_v  = data_val.y_multi.numpy()
    
    model_edg.to('cpu'); d_val_gpu.to('cpu'); torch.cuda.empty_cache()
    
    # Stratified subsample
    rng = np.random.default_rng(SEED)
    idx0 = np.where(bin_labels_v == 0)[0]
    idx1 = np.where(bin_labels_v == 1)[0]
    n_each = min(EDGE_VIZ_N // 2, len(idx0), len(idx1))
    sub_idx = np.concatenate([
        rng.choice(idx0, n_each, replace=False),
        rng.choice(idx1, n_each, replace=False),
    ])
    e_repr_sub   = e_repr[sub_idx]
    bin_sub      = bin_labels_v[sub_idx]
    mc_sub       = mc_labels_v[sub_idx]
    print(f'Subsampled {len(sub_idx):,} edges (Normal={n_each:,}, Attack={n_each:,})')
    
    # Try UMAP first, fall back to t-SNE
    try:
        import umap
        print('Using UMAP...')
        reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=30, min_dist=0.1)
        emb_2d_e = reducer.fit_transform(e_repr_sub)
        method_name = 'UMAP'
    except ImportError:
        print('UMAP not available — using t-SNE...')
        tsne_e   = TSNE(n_components=2, random_state=SEED, perplexity=40, n_iter=500)
        emb_2d_e = tsne_e.fit_transform(e_repr_sub)
        method_name = 't-SNE'
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Binary label
    for label_val, label_name, c in [(0, 'Normal', '#4CAF50'), (1, 'Attack', '#F44336')]:
        mask = (bin_sub == label_val)
        axes[0].scatter(emb_2d_e[mask, 0], emb_2d_e[mask, 1],
                        s=3, alpha=0.4, c=c, label=label_name)
    axes[0].set(title=f'Edge Repr {method_name} — Binary (10k, val)',
                xlabel=f'{method_name} 1', ylabel=f'{method_name} 2')
    axes[0].legend(markerscale=4)
    
    # Multiclass label
    mc_palette = plt.cm.tab10(np.linspace(0, 1, N_CLASSES_MULTI))
    for ci, cname in enumerate(CLASS_NAMES):
        mask = (mc_sub == ci)
        if mask.sum() == 0:
            continue
        axes[1].scatter(emb_2d_e[mask, 0], emb_2d_e[mask, 1],
                        s=3, alpha=0.5, color=mc_palette[ci], label=cname)
    axes[1].set(title=f'Edge Repr {method_name} — Multiclass (10k, val)',
                xlabel=f'{method_name} 1', ylabel=f'{method_name} 2')
    axes[1].legend(markerscale=4, fontsize=7)
    
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_edge_embeddings_tsne.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 30 — E: GAT Attention Weight Analysis
    # Per-head attention distribution + top-attention edges per attack class
    # ─────────────────────────────────────────────────────────────────────────────
    model_gat = trained_models_bin['gat'].to(DEVICE)
    d_val_gpu  = data_val.clone().to(DEVICE)
    
    model_gat.eval()
    with torch.no_grad():
        _, (att1_ret, att2_ret) = model_gat(d_val_gpu, return_attention=True)
    
    # att1_ret = (edge_index_1, alpha_1): alpha_1.shape = [E, heads]
    _, alpha1 = att1_ret
    _, alpha2 = att2_ret
    alpha1_np = alpha1.cpu().numpy()  # [E_val, heads1]
    alpha2_np = alpha2.cpu().numpy()  # [E_val, 1]
    bin_labels_v = d_val_gpu.y.cpu().numpy()
    mc_labels_v  = d_val_gpu.y_multi.cpu().numpy()
    
    model_gat.to('cpu'); d_val_gpu.to('cpu'); torch.cuda.empty_cache()
    
    n_heads = alpha1_np.shape[1]
    fig, axes = plt.subplots(2, max(n_heads, 2), figsize=(14, 6))
    
    # Row 0: Per-head attention weight distributions (Normal vs Attack)
    for h in range(n_heads):
        ax = axes[0, h]
        ax.hist(alpha1_np[bin_labels_v == 0, h], bins=50, alpha=0.6,
                density=True, color='#4CAF50', label='Normal')
        ax.hist(alpha1_np[bin_labels_v == 1, h], bins=50, alpha=0.6,
                density=True, color='#F44336', label='Attack')
        ax.set(title=f'Conv1 Head {h+1}', xlabel='Attention weight', ylabel='Density')
        ax.legend(fontsize=7)
    
    # Row 1: Mean attention per attack class
    mean_attn_per_class = []
    for ci in range(N_CLASSES_MULTI):
        mask = (mc_labels_v == ci)
        if mask.sum() == 0:
            mean_attn_per_class.append(np.zeros(n_heads))
        else:
            mean_attn_per_class.append(alpha1_np[mask].mean(axis=0))
    mean_attn_arr = np.array(mean_attn_per_class)  # [K, heads]
    
    for h in range(n_heads):
        ax = axes[1, h]
        ax.bar(range(N_CLASSES_MULTI), mean_attn_arr[:, h],
               color=[plt.cm.tab10(i / N_CLASSES_MULTI) for i in range(N_CLASSES_MULTI)], alpha=0.85)
        ax.set(title=f'Mean Attn (Head {h+1}) by Class',
               xticks=range(N_CLASSES_MULTI))
        ax.set_xticklabels([c[:4] for c in CLASS_NAMES], rotation=45, fontsize=7)
    
    plt.suptitle('GAT Attention Weight Analysis (Val Set)', y=1.01)
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_gat_attention.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')
    print(f'Mean attention weight — Normal: {alpha1_np[bin_labels_v == 0].mean():.4f}, '
          f'Attack: {alpha1_np[bin_labels_v == 1].mean():.4f}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 31 — F: Ablation Studies
    # (a) No edge features  (b) No node features  (c) 1-layer vs 2-layer GNN
    # ─────────────────────────────────────────────────────────────────────────────
    # Use best binary variant's config
    abl_variant = best_bin_variant
    abl_cfg     = best_configs[abl_variant]
    ABL_EPOCHS  = 30  # quick ablation training
    
    def run_ablation(variant, cfg, data_train_mod, data_val_mod, label):
        """Train for ABL_EPOCHS, return best val F1."""
        torch.manual_seed(SEED)
        model = build_model(variant, num_classes=1,
                            hidden_dim=cfg['hidden_dim'], dropout=cfg['dropout']).to(DEVICE)
        crit  = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
        opt   = AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
        best_f1 = -1.0
        d_tr = data_train_mod.to(DEVICE)
        d_vl = data_val_mod.to(DEVICE)
        for ep in range(ABL_EPOCHS):
            train_epoch(model, d_tr, opt, crit, 'binary')
            vm, _, _, _ = evaluate_gnn(model, d_vl, 'binary', 'val', label)
            if vm['f1'] > best_f1:
                best_f1 = vm['f1']
        d_tr.to('cpu'); d_vl.to('cpu')
        torch.cuda.empty_cache()
        return float(best_f1)
    
    ablation_results = {}
    
    # Baseline (full features)
    baseline_abl_f1 = gnn_results[f'nids_{abl_variant}_binary']['val']['f1']
    ablation_results['full_features'] = baseline_abl_f1
    print(f'Baseline ({abl_variant}) val F1: {baseline_abl_f1:.4f}')
    
    # (a) Ablate edge features — zero out edge_attr
    print('\n(a) Ablation: edge_attr = zeros')
    dt_no_edge = data_train.clone()
    dv_no_edge = data_val.clone()
    dt_no_edge.edge_attr = torch.zeros_like(dt_no_edge.edge_attr)
    dv_no_edge.edge_attr = torch.zeros_like(dv_no_edge.edge_attr)
    ablation_results['no_edge_features'] = run_ablation(abl_variant, abl_cfg,
                                                        dt_no_edge, dv_no_edge, 'no-edge-feat')
    print(f'  → val F1={ablation_results["no_edge_features"]:.4f}')
    
    # (b) Ablate node features — zero out x
    print('\n(b) Ablation: node x = zeros')
    dt_no_node = data_train.clone()
    dv_no_node = data_val.clone()
    dt_no_node.x = torch.zeros_like(dt_no_node.x)
    dv_no_node.x = torch.zeros_like(dv_no_node.x)
    ablation_results['no_node_features'] = run_ablation(abl_variant, abl_cfg,
                                                        dt_no_node, dv_no_node, 'no-node-feat')
    print(f'  → val F1={ablation_results["no_node_features"]:.4f}')
    
    # (c) 1-layer GCN vs 2-layer GCN
    print('\n(c) Ablation: 1-layer GCN vs 2-layer GCN')
    
    class NIDS_GCN_1Layer(nn.Module):
        """Single-layer GCN for ablation."""
        def __init__(self):
            super().__init__()
            hd = abl_cfg['hidden_dim']
            do = abl_cfg['dropout']
            self.conv1    = GCNConv(NODE_FEAT_DIM, hd)
            self.drop     = nn.Dropout(do)
            self.edge_mlp = _build_edge_mlp(2 * hd + EDGE_FEAT_DIM, 1, do)
        def forward(self, data):
            h = self.drop(F.relu(self.conv1(data.x, data.edge_index)))
            e = torch.cat([h[data.edge_index[0]], data.edge_attr, h[data.edge_index[1]]], dim=-1)
            return self.edge_mlp(e)
    
    # Quick train of 1-layer GCN
    torch.manual_seed(SEED)
    m1 = NIDS_GCN_1Layer().to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT.to(DEVICE))
    opt  = AdamW(m1.parameters(), lr=abl_cfg['lr'], weight_decay=1e-4)
    best_f1_1l = -1.0
    d_tr = data_train.clone().to(DEVICE); d_vl = data_val.clone().to(DEVICE)
    for ep in range(ABL_EPOCHS):
        train_epoch(m1, d_tr, opt, crit, 'binary')
        vm, _, _, _ = evaluate_gnn(m1, d_vl, 'binary', 'val', '1-layer-gcn')
        if vm['f1'] > best_f1_1l: best_f1_1l = vm['f1']
    ablation_results['gcn_1layer'] = float(best_f1_1l)
    ablation_results['gcn_2layer'] = gnn_results['nids_gcn_binary']['val']['f1']
    d_tr.to('cpu'); d_vl.to('cpu'); torch.cuda.empty_cache()
    print(f'  1-layer GCN val F1={best_f1_1l:.4f} | 2-layer GCN val F1={ablation_results["gcn_2layer"]:.4f}')
    
    eval_suite['ablation'] = ablation_results
    
    # ── Plot ablation summary ────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 4))
    abl_labels = ['Full features', 'No edge feat', 'No node feat', 'GCN 1-layer', 'GCN 2-layer']
    abl_values = [ablation_results[k] for k in
                  ['full_features', 'no_edge_features', 'no_node_features', 'gcn_1layer', 'gcn_2layer']]
    bar_c = ['#4CAF50'] + ['#F44336'] * 2 + ['#2196F3', '#2196F3']
    ax.bar(abl_labels, abl_values, color=bar_c, alpha=0.85)
    ax.axhline(baseline_abl_f1, linestyle='--', color='gray', alpha=0.6, label='Full baseline')
    ax.set(title=f'Ablation Study — NIDS-{abl_variant.upper()} Binary (Val F1)',
           ylabel='Validation F1', ylim=[0, 1])
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    for rect, v in zip(ax.patches, abl_values):
        ax.text(rect.get_x() + rect.get_width()/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=8)
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_ablation.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 32 — G: Robustness, Homophily, Degree-stratified Performance
    # ─────────────────────────────────────────────────────────────────────────────
    model_rob = trained_models_bin[best_bin_variant].to(DEVICE)
    model_rob.eval()
    
    # ── 1. DropEdge robustness ────────────────────────────────────────────────────
    drop_edge_rates = [0.0, 0.1, 0.2, 0.3, 0.5]
    drop_edge_f1    = []
    
    for p in drop_edge_rates:
        d_perturb = data_val.clone().to(DEVICE)
        if p > 0:
            E = d_perturb.num_edges
            keep_mask = torch.rand(E) > p
            d_perturb.edge_index = d_perturb.edge_index[:, keep_mask]
            d_perturb.edge_attr  = d_perturb.edge_attr[keep_mask]
            d_perturb.y          = d_perturb.y[keep_mask]
            d_perturb.y_multi    = d_perturb.y_multi[keep_mask]
        with torch.no_grad():
            out_r = model_rob(d_perturb)
        probs_r = torch.sigmoid(out_r.squeeze(-1)).cpu().numpy()
        preds_r = (probs_r > 0.5).astype(int)
        lbl_r   = d_perturb.y.cpu().numpy()
        drop_edge_f1.append(f1_score(lbl_r, preds_r, zero_division=0))
        d_perturb.to('cpu')
    
    # ── 2. Feature noise robustness ───────────────────────────────────────────────
    noise_levels = [0.0, 0.01, 0.05, 0.1, 0.2]
    noise_f1     = []
    
    for sigma in noise_levels:
        d_noisy = data_val.clone().to(DEVICE)
        if sigma > 0:
            noise = torch.randn_like(d_noisy.edge_attr) * sigma
            d_noisy.edge_attr = d_noisy.edge_attr + noise
        with torch.no_grad():
            out_n = model_rob(d_noisy)
        probs_n = torch.sigmoid(out_n.squeeze(-1)).cpu().numpy()
        preds_n = (probs_n > 0.5).astype(int)
        lbl_n   = data_val.y.numpy()
        noise_f1.append(f1_score(lbl_n, preds_n, zero_division=0))
        d_noisy.to('cpu')
    
    # ── 3. Edge label homophily ratio ─────────────────────────────────────────────
    # Fraction of edges where (src_label == dst_label) in a node-aggregated sense
    # Since edges are labelled (not nodes), compute: frac of attack edges whose
    # source node has src_attack_ratio > 0.5
    attack_mask = data_val.y.numpy() == 1
    src_nodes_of_attack = data_val.edge_index[0][data_val.y == 1].numpy()
    node_atk_ratio = data_train.x[:, -2].numpy()  # src_attack_ratio column (index from node_feat_cols)
    # Find which column is src_attack_ratio
    try:
        src_col_idx = node_meta['node_feat_cols'].index('src_attack_ratio')
    except ValueError:
        src_col_idx = -2  # fallback
    src_atk_ratio_per_edge = data_train.x[src_nodes_of_attack, src_col_idx].numpy()
    homophily = float((src_atk_ratio_per_edge > 0.3).mean()) if len(src_atk_ratio_per_edge) > 0 else 0.0
    eval_suite['robustness']['edge_homophily_ratio'] = homophily
    print(f'Edge homophily: {homophily:.4f} '
          f'(fraction of attack edges sourced from high-attack-ratio IPs)')
    
    # ── 4. Degree-stratified performance ─────────────────────────────────────────
    out_deg = torch.bincount(data_val.edge_index[0], minlength=N_NODES).numpy()
    d_val_gpu = data_val.clone().to(DEVICE)
    with torch.no_grad():
        out_all = model_rob(d_val_gpu)
    probs_all = torch.sigmoid(out_all.squeeze(-1)).cpu().numpy()
    preds_all = (probs_all > 0.5).astype(int)
    labels_all = data_val.y.numpy()
    src_node_deg_per_edge = out_deg[data_val.edge_index[0].numpy()]
    d_val_gpu.to('cpu'); torch.cuda.empty_cache()
    
    degree_buckets = [(1, 1000), (1000, 5000), (5000, 15000), (15000, 50000)]
    degree_f1 = {}
    for lo, hi in degree_buckets:
        mask = (src_node_deg_per_edge >= lo) & (src_node_deg_per_edge < hi)
        if mask.sum() > 10:
            f1_v = f1_score(labels_all[mask], preds_all[mask], zero_division=0)
            degree_f1[f'{lo}-{hi}'] = float(f1_v)
            print(f'  Degree [{lo:>6},{hi:>6}): {mask.sum():>8,} edges | F1={f1_v:.4f}')
    
    eval_suite['robustness']['drop_edge'] = dict(zip([str(p) for p in drop_edge_rates], drop_edge_f1))
    eval_suite['robustness']['feature_noise'] = dict(zip([str(s) for s in noise_levels], noise_f1))
    eval_suite['robustness']['degree_stratified_f1'] = degree_f1
    model_rob.to('cpu')
    
    # ── Plot robustness ───────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].plot(drop_edge_rates, drop_edge_f1, 'o-', color='#2196F3', linewidth=2)
    axes[0].set(title='DropEdge Robustness (Val)',
                xlabel='Fraction of edges removed', ylabel='F1 Score')
    axes[0].grid(alpha=0.3)
    
    axes[1].plot(noise_levels, noise_f1, 's-', color='#FF5722', linewidth=2)
    axes[1].set(title='Feature Noise Robustness (Val)',
                xlabel='Gaussian noise σ on edge_attr', ylabel='F1 Score')
    axes[1].grid(alpha=0.3)
    
    if degree_f1:
        dkeys = list(degree_f1.keys())
        axes[2].bar(range(len(dkeys)), [degree_f1[k] for k in dkeys],
                    color='#4CAF50', alpha=0.85)
        axes[2].set(title='F1 by Source Node Out-Degree Bucket',
                    xlabel='Degree range', ylabel='F1 Score', xticks=range(len(dkeys)))
        axes[2].set_xticklabels(dkeys, rotation=25)
        axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    save_p = os.path.join(OUT_DIR, 'gnn_robustness.png')
    plt.savefig(save_p, bbox_inches='tight'); plt.show()
    print(f'Saved → {save_p}')

In [ ]:
if RUN_EVAL_SUITE:
    # ─────────────────────────────────────────────────────────────────────────────
    # Cell 33 — H: Gradient Attribution + (Optional) GNNExplainer
    # Identifies which edge features most influence attack predictions
    # ─────────────────────────────────────────────────────────────────────────────
    
    # ── Method 1: Gradient × Input attribution ───────────────────────────────────
    # For K representative edges (one per rare attack class on val), compute
    # |gradient of output logit w.r.t. edge_attr| as feature importance.
    
    model_attr = trained_models_bin[best_bin_variant].to(DEVICE)
    model_attr.eval()
    
    d_val_gpu = data_val.clone().to(DEVICE)
    with torch.no_grad():
        out_v = model_attr(d_val_gpu)
    probs_v = torch.sigmoid(out_v.squeeze(-1)).cpu().numpy()
    preds_v = (probs_v > 0.5).astype(int)
    labels_v = data_val.y.cpu().numpy()
    mc_labels_v = data_val.y_multi.cpu().numpy()
    
    # Find one misclassified attack edge per rare attack class
    RARE_CLASSES = {0: 'Analysis', 1: 'Backdoor', 2: 'DoS', 8: 'Shellcode', 9: 'Worms'}
    target_edge_indices = {}
    for ci, cname in RARE_CLASSES.items():
        # Edges of this class that were misclassified as normal
        cand = np.where((mc_labels_v == ci) & (preds_v == 0))[0]
        if len(cand) == 0:
            # Fall back to any edge of this class
            cand = np.where(mc_labels_v == ci)[0]
        if len(cand) > 0:
            target_edge_indices[cname] = int(cand[0])
    
    print(f'Target edges for attribution: {target_edge_indices}')
    
    # Compute gradient attribution
    all_attributions = {}
    for cname, edge_idx in target_edge_indices.items():
        # Clone and enable gradients on edge_attr
        d_copy = data_val.clone().to(DEVICE)
        ea_copy = d_copy.edge_attr.detach().clone().requires_grad_(True)
        d_copy.edge_attr = ea_copy
    
        out_g = model_attr(d_copy)     # [E, 1]
        logit = out_g[edge_idx, 0]
        logit.backward()
    
        grad  = ea_copy.grad[edge_idx].detach().cpu().numpy()   # [37]
        inp   = data_val.edge_attr[edge_idx].cpu().numpy()      # [37]
        attr  = np.abs(grad * inp)                              # gradient × input
        all_attributions[cname] = attr
        d_copy.to('cpu')
    
    model_attr.to('cpu'); d_val_gpu.to('cpu'); torch.cuda.empty_cache()
    
    # ── Plot attribution heatmap ──────────────────────────────────────────────────
    n_attack_classes = len(all_attributions)
    if n_attack_classes > 0:
        attr_matrix = np.array([all_attributions[c] for c in all_attributions])
        fig, ax = plt.subplots(figsize=(16, max(3, n_attack_classes)))
        im = ax.imshow(attr_matrix, aspect='auto', cmap='YlOrRd')
        ax.set_yticks(range(n_attack_classes))
        ax.set_yticklabels(list(all_attributions.keys()))
        ax.set_xticks(range(len(FEATURE_NAMES)))
        ax.set_xticklabels(FEATURE_NAMES, rotation=60, ha='right', fontsize=7)
        ax.set(title=f'Gradient×Input Attribution — NIDS-{best_bin_variant.upper()} (per rare attack class)',
               xlabel='Edge Feature', ylabel='Attack Class')
        plt.colorbar(im, ax=ax, label='|grad × input|')
        plt.tight_layout()
        save_p = os.path.join(OUT_DIR, 'gnn_explainer_attributions.png')
        plt.savefig(save_p, bbox_inches='tight'); plt.show()
        print(f'Saved → {save_p}')
    
        # Top-3 features per class
        print('\nTop-3 most influential features per attack class:')
        for cname, attr in all_attributions.items():
            top3 = np.argsort(attr)[::-1][:3]
            print(f'  {cname:<14}: {[FEATURE_NAMES[i] for i in top3]}')
    
    # ── Optional: PyG GNNExplainer ───────────────────────────────────────────────
    try:
        from torch_geometric.explain import Explainer, GNNExplainer
    
        # GNNExplainer requires a model wrapper compatible with the Explainer API
        # We use a simple edge-to-prediction wrapper for one target edge
        print('\nGNNExplainer available — skipping full run (computationally intensive on 1.6M edge graph).')
        print('To run: set USE_GNNEXPLAINER=True and call explainer() on a single edge index.')
        USE_GNNEXPLAINER = False  # Set True to run (adds ~5–10 min per edge)
        # Full implementation available in src/explain.py
    except ImportError:
        print('GNNExplainer not available in this PyG version.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 34 — Save gnn_results.json + gnn_eval_suite.json
# ─────────────────────────────────────────────────────────────────────────────

# ── gnn_results.json (matches baseline_results.json schema) ──────────────────
gnn_results_path = os.path.join(OUT_DIR, 'gnn_results.json')
with open(gnn_results_path, 'w') as f:
    json.dump(_serialisable(gnn_results), f, indent=2)
print(f'Saved → {gnn_results_path}')

# ── gnn_eval_suite.json ───────────────────────────────────────────────────────
eval_suite['hpo_results'] = _serialisable(hpo_results)
eval_suite['best_configs'] = _serialisable(best_configs)
eval_suite['best_binary_variant']     = best_bin_variant
eval_suite['best_multiclass_variant'] = best_mc_variant

suite_path = os.path.join(OUT_DIR, 'gnn_eval_suite.json')
with open(suite_path, 'w') as f:
    json.dump(_serialisable(eval_suite), f, indent=2)
print(f'Saved → {suite_path}')

# ── Print checkpoint inventory ────────────────────────────────────────────────
print('\nCheckpoints saved:')
for variant in ['gcn', 'gat', 'sage', 'full']:
    for task in ['binary', 'multiclass']:
        p = os.path.join(OUT_DIR, f'nids_{variant}_{task}.pt')
        status = '✓' if os.path.exists(p) else '✗ MISSING'
        print(f'  {status}  nids_{variant}_{task}.pt')

# ── Verification: round-trip load best binary model ──────────────────────────
print('\nVerifying checkpoint load...')
chk_path = os.path.join(OUT_DIR, f'nids_{best_bin_variant}_binary.pt')
chk_cfg  = best_configs[best_bin_variant]
model_verify = build_model(best_bin_variant, num_classes=1,
                           hidden_dim=chk_cfg['hidden_dim'],
                           dropout=chk_cfg['dropout'])
model_verify.load_state_dict(torch.load(chk_path, map_location='cpu', weights_only=True))
model_verify.eval()
print(f'  NIDS-{best_bin_variant.upper()} binary checkpoint loads OK.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 35 — Generate phase5_results.md + update outputs.md (local only)
# ─────────────────────────────────────────────────────────────────────────────
def fmt(d, k, decimals=4):
    v = d.get(k, float('nan'))
    return f'{v:.{decimals}f}' if isinstance(v, (int, float)) and not (isinstance(v, float) and v != v) else '—'

# Identify best models
best_bin_test_f1 = max(gnn_results[f'nids_{v}_binary']['test']['f1'] for v in ['gcn','gat','sage','full'])
best_mc_test_f1  = max(gnn_results[f'nids_{v}_multiclass']['test']['f1_macro'] for v in ['gcn','gat','sage','full'])

lines = []
lines.append('# Phase 5 — GNN Training Results\n\n')
lines.append('## Binary Classification Results\n\n')
lines.append('### Val Set (attack ratio 4.84%)\n\n')
lines.append('| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR | MCC | PR-AUC |\n')
lines.append('|---|---|---|---|---|---|---|---|---|\n')
for v in ['gcn', 'gat', 'sage', 'full']:
    key = f'nids_{v}_binary'; val = gnn_results[key]['val']
    lines.append(f'| NIDS-{v.upper()} | {fmt(val,"accuracy")} | {fmt(val,"precision")} | '
                 f'{fmt(val,"recall")} | {fmt(val,"f1")} | {fmt(val,"roc_auc")} | '
                 f'{fmt(val,"fpr")} | {fmt(val,"mcc")} | {fmt(val,"pr_auc")} |\n')
lines.append('\n### Test Set (attack ratio 55.06%)\n\n')
lines.append('| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR | MCC | PR-AUC |\n')
lines.append('|---|---|---|---|---|---|---|---|---|\n')
for v in ['gcn', 'gat', 'sage', 'full']:
    key = f'nids_{v}_binary'; tst = gnn_results[key]['test']
    lines.append(f'| NIDS-{v.upper()} | {fmt(tst,"accuracy")} | {fmt(tst,"precision")} | '
                 f'{fmt(tst,"recall")} | {fmt(tst,"f1")} | {fmt(tst,"roc_auc")} | '
                 f'{fmt(tst,"fpr")} | {fmt(tst,"mcc")} | {fmt(tst,"pr_auc")} |\n')
lines.append('\n## Multiclass Classification Results\n\n')
lines.append('### Val Set\n\n')
lines.append('| Model | Accuracy | F1-Macro | F1-Weighted |\n')
lines.append('|---|---|---|---|\n')
for v in ['gcn', 'gat', 'sage', 'full']:
    key = f'nids_{v}_multiclass'; val = gnn_results[key]['val']
    lines.append(f'| NIDS-{v.upper()} | {fmt(val,"accuracy")} | {fmt(val,"f1_macro")} | {fmt(val,"f1_weighted")} |\n')
lines.append('\n### Test Set\n\n')
lines.append('| Model | Accuracy | F1-Macro | F1-Weighted |\n')
lines.append('|---|---|---|---|\n')
for v in ['gcn', 'gat', 'sage', 'full']:
    key = f'nids_{v}_multiclass'; tst = gnn_results[key]['test']
    lines.append(f'| NIDS-{v.upper()} | {fmt(tst,"accuracy")} | {fmt(tst,"f1_macro")} | {fmt(tst,"f1_weighted")} |\n')
lines.append(f'\n## Best Models\n\n')
lines.append(f'- **Best Binary GNN:** NIDS-{best_bin_variant.upper()} | Test F1={best_bin_test_f1:.4f}\n')
lines.append(f'- **Best Multiclass GNN:** NIDS-{best_mc_variant.upper()} | Test F1-macro={best_mc_test_f1:.4f}\n')

# Baseline comparison
lb_bin  = max(baseline_results[m]['test']['f1'] for m in ['lr_binary','rf_binary','xgb_binary','lgbm_binary','mlp_binary'])
lb_mc   = max(baseline_results[m]['test']['f1_macro'] for m in ['lr_multiclass','rf_multiclass','xgb_multiclass','lgbm_multiclass','mlp_multiclass'])
lines.append(f'- **GNN vs best baseline (binary):** ΔF1={best_bin_test_f1 - lb_bin:+.4f}\n')
lines.append(f'- **GNN vs best baseline (multiclass):** ΔF1-macro={best_mc_test_f1 - lb_mc:+.4f}\n')

# HPO config
lines.append('\n## Best Hyperparameters\n\n')
lines.append('| Variant | hidden_dim | dropout | lr |\n')
lines.append('|---|---|---|---|\n')
for v in ['gcn', 'gat', 'sage', 'full']:
    c = best_configs[v]
    lines.append(f'| NIDS-{v.upper()} | {c["hidden_dim"]} | {c["dropout"]} | {c["lr"]} |\n')

# Eval suite summary
lines.append('\n## Evaluation Suite Summary\n\n')
if 'calibration' in eval_suite and best_bin_variant in eval_suite['calibration']:
    ec = eval_suite['calibration'][best_bin_variant]
    lines.append(f'- **Calibration** (NIDS-{best_bin_variant.upper()}, test): '
                 f'ECE={ec["test"]["ece"]:.4f}, Brier={ec["test"]["brier"]:.4f}\n')
if 'uncertainty' in eval_suite and best_bin_variant in eval_suite['uncertainty']:
    uc = eval_suite['uncertainty'][best_bin_variant]
    lines.append(f'- **MC Dropout** ({T_MC} passes, val): '
                 f'MC-F1={uc["mc_f1"]:.4f} vs Det-F1={uc["det_f1"]:.4f}, '
                 f'Uncertainty (correct)={uc["mean_std_correct"]:.4f}, '
                 f'(incorrect)={uc["mean_std_incorrect"]:.4f}\n')
if 'ablation' in eval_suite:
    abl = eval_suite['ablation']
    lines.append(f'- **Ablation** (val F1): full={abl.get("full_features",0):.4f}, '
                 f'no-edge-feat={abl.get("no_edge_features",0):.4f}, '
                 f'no-node-feat={abl.get("no_node_features",0):.4f}\n')

# Save phase5_results.md
md_path = os.path.join(OUT_DIR, 'phase5_results.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)
print(f'Saved → {md_path}')

# ── Update outputs.md (local only) ───────────────────────────────────────────
if PLATFORM == 'local':
    outputs_md_path = os.path.join(OUT_MDS, 'outputs.md')
    try:
        with open(outputs_md_path, 'r', encoding='utf-8') as f:
            content = f.read()

        # Build Phase 5 replacement block
        p5_block_lines = [
            '## § Phase 5 — GNN Model Training\n\n',
            '> Status: ✅ Complete\n\n',
            '### 5.1 Binary Classification Results — Val Set (attack ratio 4.84%)\n\n',
            '| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR | MCC |\n',
            '|---|---|---|---|---|---|---|---|\n',
        ]
        for v in ['gcn', 'gat', 'sage', 'full']:
            key = f'nids_{v}_binary'; val = gnn_results[key]['val']
            p5_block_lines.append(
                f'| NIDS-{v.upper()} | {fmt(val,"accuracy")} | {fmt(val,"precision")} | '
                f'{fmt(val,"recall")} | {fmt(val,"f1")} | {fmt(val,"roc_auc")} | '
                f'{fmt(val,"fpr")} | {fmt(val,"mcc")} |\n')
        p5_block_lines += [
            '\n### 5.2 Binary Classification Results — Test Set (attack ratio 55.06%)\n\n',
            '| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR | MCC |\n',
            '|---|---|---|---|---|---|---|---|\n',
        ]
        for v in ['gcn', 'gat', 'sage', 'full']:
            key = f'nids_{v}_binary'; tst = gnn_results[key]['test']
            p5_block_lines.append(
                f'| NIDS-{v.upper()} | {fmt(tst,"accuracy")} | {fmt(tst,"precision")} | '
                f'{fmt(tst,"recall")} | {fmt(tst,"f1")} | {fmt(tst,"roc_auc")} | '
                f'{fmt(tst,"fpr")} | {fmt(tst,"mcc")} |\n')
        p5_block_lines += [
            '\n### 5.3 Multiclass Classification Results — Val Set\n\n',
            '| Model | Accuracy | F1-Macro | F1-Weighted |\n',
            '|---|---|---|---|\n',
        ]
        for v in ['gcn', 'gat', 'sage', 'full']:
            key = f'nids_{v}_multiclass'; val = gnn_results[key]['val']
            p5_block_lines.append(f'| NIDS-{v.upper()} | {fmt(val,"accuracy")} | {fmt(val,"f1_macro")} | {fmt(val,"f1_weighted")} |\n')
        p5_block_lines += [
            '\n### 5.4 Multiclass Classification Results — Test Set\n\n',
            '| Model | Accuracy | F1-Macro | F1-Weighted |\n',
            '|---|---|---|---|\n',
        ]
        for v in ['gcn', 'gat', 'sage', 'full']:
            key = f'nids_{v}_multiclass'; tst = gnn_results[key]['test']
            p5_block_lines.append(f'| NIDS-{v.upper()} | {fmt(tst,"accuracy")} | {fmt(tst,"f1_macro")} | {fmt(tst,"f1_weighted")} |\n')
        p5_block_lines += [
            f'\n### 5.5 Best GNN Models\n\n',
            f'- **Best Binary:** NIDS-{best_bin_variant.upper()} | Test F1={best_bin_test_f1:.4f} | vs best baseline RF ΔF1={best_bin_test_f1 - lb_bin:+.4f}\n',
            f'- **Best Multiclass:** NIDS-{best_mc_variant.upper()} | Test F1-macro={best_mc_test_f1:.4f} | vs best baseline RF ΔF1={best_mc_test_f1 - lb_mc:+.4f}\n',
            '\n### 5.6 Outputs Saved\n\n',
        ]
        for variant in ['gcn', 'gat', 'sage', 'full']:
            for task in ['binary', 'multiclass']:
                p5_block_lines.append(f'- [x] `outputs/models/nids_{variant}_{task}.pt`\n')
        p5_block_lines += [
            '- [x] `outputs/models/gnn_results.json`\n',
            '- [x] `outputs/models/gnn_eval_suite.json`\n',
            '- [x] `outputs/models/phase5_results.md`\n',
        ]

        new_p5 = ''.join(p5_block_lines)

        # Find and replace the Phase 5 section
        import re
        pattern = r'## § Phase 5 — GNN Model Training.*?(?=## § Phase 6|\Z)'
        new_content = re.sub(pattern, new_p5, content, flags=re.DOTALL)

        if new_content == content:
            # Fallback: append
            new_content = content + '\n' + new_p5

        with open(outputs_md_path, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print(f'outputs.md updated → {outputs_md_path}')
    except FileNotFoundError:
        print(f'outputs.md not found at {outputs_md_path} — skipping update.')
else:
    print(f'Platform={PLATFORM}: outputs.md update skipped. Download phase5_results.md and update manually.')
    print(f'  phase5_results.md saved at: {md_path}')

print('\n✓ Phase 5 complete!')
print(f'  Best binary GNN  : NIDS-{best_bin_variant.upper()} | test F1={best_bin_test_f1:.4f} (baseline={lb_bin:.4f}, Δ={best_bin_test_f1-lb_bin:+.4f})')
print(f'  Best multiclass  : NIDS-{best_mc_variant.upper()} | test F1-macro={best_mc_test_f1:.4f} (baseline={lb_mc:.4f}, Δ={best_mc_test_f1-lb_mc:+.4f})')